# Menjelaskan prediksi teks per dokumen: kata mana yang mendorong tebakan?
**Kapan dipakai:** untuk slide "kenapa model menebak begitu" dan untuk mencari pola kesalahan.
Bobot kata global per kelas sudah ada di `arsenal/teks/klasifikasi/tfidf` #4; notebook ini
menjelaskan **satu dokumen** dan berlaku untuk **model apa pun**:

1. **Oklusi (leave-one-word-out):** hapus satu kata → log-odds kelas tebakan turun berapa?
   Positif = kata itu mendukung tebakan. Butuh hanya fungsi `predict_proba(daftar_teks)`.
   Log-odds (bukan peluang) supaya contoh yang sangat yakin (peluang 0,999) tetap terbaca.
2. **Kata penentu (kontrafaktual):** hapus kata pendukung terkuat satu per satu sampai tebakan
   berbalik. "Hapus 'tidak' saja → negatif jadi positif" = bahan slide yang mudah dicerna.
3. **Peringkat kata gabungan:** rata-rata oklusi di ratusan dokumen validasi (memperhitungkan
   konteks, beda dari bobot global).

Contoh diambil dari lipatan validasi (model tidak pernah melihatnya): benar-yakin, salah-yakin,
ragu. Semua hasil (PNG, HTML berwarna, CSV) disimpan di `OUTPUT_FOLDER`. Catatan: HTML/CSV
memuat teks asli data; cek aturan panitia sebelum memindahkannya dari PC lomba.

In [ ]:
import html
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline, make_union

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/penjelasan_teks"
TEXT_COLS = ["text"]          # GANTI saat lomba
TARGET = "label"              # GANTI saat lomba

N_FOLDS = 5
SEED = 42
MIN_WORDS = 5                 # contoh yang ditampilkan: minimal sekian kata
MAX_WORDS = 60                # ... dan maksimal sekian kata (supaya terbaca)
PER_KIND = 2                  # contoh per jenis (benar-yakin, salah-yakin, ragu)
N_AGGREGATE = 300             # dokumen validasi untuk peringkat kata gabungan
MIN_COUNT = 5                 # kata harus muncul di >= sekian dokumen agar masuk peringkat
TOP_WORDS = 10

## Baca data
Kolom teks digabung; spasi dirapikan karena "kata" = potongan di antara spasi.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
texts = train[TEXT_COLS].fillna("").astype(str)
texts = texts.agg(" . ".join, axis=1)
texts = texts.str.split().str.join(" ")
labels = train[TARGET].astype(str)
CLASSES = np.unique(labels)
y = np.searchsorted(CLASSES, labels.values)
print("baris:", len(train), "| kelas:", CLASSES)

## Model yang dijelaskan + `predict_proba`
Default: TF-IDF kata + huruf → LogReg, dilatih di lipatan 0 (lipatan sama dengan notebook
OOF lain). **Model lain:** cukup ganti isi `predict_proba` dengan fungsi yang menerima daftar
teks dan mengembalikan peluang n × jumlah kelas (urutan kelas `CLASSES`), mis. pipeline
`klasifikasi_embedding` atau model fine-tune yang masih ada di memori.

In [ ]:
def make_model():
    words = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    chars = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)
    return make_pipeline(make_union(words, chars), LogisticRegression(C=4.0, max_iter=2000))


splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
idx_fit, idx_val = list(splitter.split(texts, y))[0]
model = make_model().fit(texts.iloc[idx_fit], y[idx_fit])


def predict_proba(text_list):
    return model.predict_proba(list(text_list))


def log_odds(proba):
    proba = np.clip(proba, 1e-6, 1 - 1e-6)
    return np.log(proba / (1 - proba))


val_texts = texts.iloc[idx_val].reset_index(drop=True)
val_y = y[idx_val]
val_proba = predict_proba(val_texts)
val_pred = val_proba.argmax(axis=1)
print("F1 makro lipatan 0:", round(f1_score(val_y, val_pred, average="macro"), 4))

## Oklusi + kata penentu
`occlusion`: satu panggilan model untuk semua varian teks (cepat), hasil = turunnya log-odds
`log(p / (1 − p))` kelas tebakan. `flip_words`: hapus kata
pendukung terkuat berurutan sampai kelas tebakan berubah (maks. 8 kata; kosong = tidak berbalik).

In [ ]:
def occlusion(text, class_id):
    words = text.split()
    variants = []
    for position in range(len(words)):
        variants.append(" ".join(words[:position] + words[position + 1:]))
    if len(variants) == 0:
        return words, np.zeros(0)
    base = log_odds(predict_proba([text])[0, class_id])
    return words, base - log_odds(predict_proba(variants)[:, class_id])


def flip_words(words, drops, class_id, max_steps=8):
    removed = []
    for position in np.argsort(-drops)[:max_steps]:
        removed.append(position)
        kept = []
        for index, word in enumerate(words):
            if index not in removed:
                kept.append(word)
        if predict_proba([" ".join(kept)])[0].argmax() != class_id:
            return np.array(words)[removed].tolist()
    return []


demo_words, demo_drops = occlusion("makanannya tidak enak dan mahal", 0)
print(dict(zip(demo_words, demo_drops.round(3).tolist())), flip_words(demo_words, demo_drops, 0))

## Pilih contoh: benar-yakin, salah-yakin, ragu
Ragu = selisih peluang dua kelas teratas paling kecil. Hanya teks `MIN_WORDS`–`MAX_WORDS` kata.

In [ ]:
confidence = val_proba.max(axis=1)
ordered = np.sort(val_proba, axis=1)
margin = ordered[:, -1] - ordered[:, -2]
n_words = val_texts.str.split().str.len()
short = (n_words.to_numpy() >= MIN_WORDS) & (n_words.to_numpy() <= MAX_WORDS)
correct = val_pred == val_y
kinds = {"benar-yakin": np.where(short & correct, -confidence, np.inf),
         "salah-yakin": np.where(short & ~correct, -confidence, np.inf),
         "ragu": np.where(short, margin, np.inf)}
examples = []
for kind, order_key in kinds.items():
    for row in np.argsort(order_key)[:PER_KIND]:
        words, drops = occlusion(val_texts[row], val_pred[row])
        examples.append({"jenis": kind, "baris_val": row, "benar": CLASSES[val_y[row]],
                         "tebak": CLASSES[val_pred[row]], "peluang": confidence[row],
                         "kata": words, "kontribusi": drops,
                         "kata_penentu": flip_words(words, drops, val_pred[row])})
for example in examples:
    print(example["jenis"], "| benar", example["benar"], "| tebak", example["tebak"],
          f"{example['peluang']:.2f}", "| hapus ini → berbalik:", example["kata_penentu"])

## Grafik: kata pendukung (biru) dan penentang (oranye) tiap contoh
Batang = log-odds kelas tebakan turun sebesar itu kalau kata tersebut dihapus.

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(examples) / n_cols))
figure, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4.2 * n_rows), squeeze=False)
for ax, example in zip(axes.ravel(), examples):
    top = np.argsort(-np.abs(example["kontribusi"]))[:8][::-1]
    values = example["kontribusi"][top]
    colors = np.where(values > 0, "#0072B2", "#E69F00")
    ax.barh(np.array(example["kata"])[top], values, color=colors)
    ax.axvline(0, color="#1a1a1a", linewidth=0.8)
    ax.set_xlabel("turunnya log-odds bila kata dihapus")
    ax.set_title(f"{example['jenis']}: benar {example['benar']}, tebak {example['tebak']}",
                 fontsize=11)
for ax in axes.ravel()[len(examples):]:
    ax.axis("off")
figure.tight_layout()
figure.savefig(f"{OUTPUT_FOLDER}/oklusi_contoh.png", dpi=200, bbox_inches="tight")
plt.show()

## HTML berwarna (buka di browser → tangkap layar untuk slide) + CSV kontribusi

In [ ]:
def highlight(words, drops):
    scale = np.abs(drops).max() + 1e-9
    spans = []
    for word, drop in zip(words, drops):
        color = "0,114,178"
        if drop < 0:
            color = "230,159,0"
        alpha = round(float(abs(drop) / scale) * 0.85, 2)
        spans.append(f'<span style="background:rgba({color},{alpha})">{html.escape(word)}</span>')
    return " ".join(spans)


parts = ["<html><meta charset='utf-8'><body style='font-family:sans-serif'>"]
rows = []
for number, example in enumerate(examples):
    parts.append(f"<h3>{example['jenis']} | benar: {example['benar']} | tebak: "
                 f"{example['tebak']} ({example['peluang']:.2f})</h3>")
    marked = highlight(example["kata"], example["kontribusi"])
    parts.append(f"<p style='line-height:2'>{marked}</p>")
    parts.append(f"<p>kata penentu: {html.escape(str(example['kata_penentu']))}</p>")
    for word, drop in zip(example["kata"], example["kontribusi"]):
        rows.append({"contoh": number, "jenis": example["jenis"], "kata": word, "kontribusi": drop})
with open(f"{OUTPUT_FOLDER}/oklusi_contoh.html", "w", encoding="utf-8") as handle:
    handle.write("\n".join(parts + ["</body></html>"]))
contribution_table = pd.DataFrame(rows).round(4)
contribution_table.to_csv(f"{OUTPUT_FOLDER}/oklusi_contoh.csv", index=False)
print("ditulis: oklusi_contoh.html, oklusi_contoh.csv")

## Peringkat kata gabungan per kelas tebakan
Oklusi di `N_AGGREGATE` dokumen validasi acak; rata-rata kontribusi tiap kata (huruf kecil)
untuk kelas yang ditebak. Kata yang muncul < `MIN_COUNT` kali dibuang (supaya tidak kebetulan).

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for row in rng.permutation(len(val_texts))[:N_AGGREGATE]:
    words, drops = occlusion(val_texts[row], val_pred[row])
    for word, drop in zip(words, drops):
        rows.append({"kelas": CLASSES[val_pred[row]], "kata": word.lower(), "kontribusi": drop})
contributions = pd.DataFrame(rows)
ranking = contributions.groupby(["kelas", "kata"])["kontribusi"].agg(["mean", "size"])
ranking = ranking[ranking["size"] >= MIN_COUNT].reset_index()
ranking = ranking[ranking["kata"].str.contains(r"[^\W\d_]")]    # buang tanda baca/angka
ranking = ranking.sort_values("mean", ascending=False)
top_words = ranking.groupby("kelas").head(TOP_WORDS)
top_words.round(4).to_csv(f"{OUTPUT_FOLDER}/kata_penentu_per_kelas.csv", index=False)
print(top_words.round(3).to_string(index=False))

Grafik peringkat (satu panel per kelas, maks. 6 kelas).

In [ ]:
shown = list(CLASSES[:6])
figure, axes = plt.subplots(1, len(shown), figsize=(5 * len(shown), 4.5), squeeze=False)
for ax, name in zip(axes[0], shown):
    part = top_words[top_words["kelas"] == name].iloc[::-1]
    ax.barh(part["kata"], part["mean"], color="#0072B2")
    ax.set_title(f"tebakan: {name}", fontsize=12)
    ax.set_xlabel("rata-rata turunnya log-odds")
figure.tight_layout()
figure.savefig(f"{OUTPUT_FOLDER}/kata_penentu_per_kelas.png", dpi=200, bbox_inches="tight")
plt.show()